In [35]:

import cv2 as cv
import os
from utils import *
from keypoints import *
import argparse
from datetime import datetime
import yaml
import shutil
import json


### prepare datasets
- convert multiple annotated projects from `red` into `jarvis` projects using the script `red3d2jarvis.py`
- collate the mutiple `jarvis` projects into a single directory (say `~/data/jarvis_merge` with projects `dataset_11_06` and `dataset_11_25`)


In [25]:
src_dir = "/Users/othayothr/data/jarvis_merge"


input_datasets = [o for o in os.listdir(src_dir) if os.path.isdir(os.path.join(src_dir, o))]

print(input_datasets)

# create parent directory where merged dataset will be created
output_dataset = "/Users/othayothr/data/merged_dataset"
if not os.path.exists(output_dataset):
    os.makedirs(output_dataset)


['dataset_11_06', 'dataset_11_25']


#### copy camera calibration files



In [ ]:
# create parent output directory for calibration files
output_calib_dir = os.path.join(output_dataset, "calib_params")
if not os.path.exists(output_calib_dir):
    os.makedirs(output_calib_dir)
    
# loop over all datasets    
for dset in input_datasets:
    
    src_calib_dir = os.path.join(src_dir, dset, "calib_params")
    all_calibs = glob.glob(src_calib_dir + "/*")
    all_calibs.sort()
    select_most_recent_calib = all_calibs[-1]
    print("Select most recent label: {}".format(select_most_recent_calib))
    src_calib_dir = select_most_recent_calib
    
    
    # create output directory for dataset and copy calibration files
    shutil.copytree(os.path.dirname(src_calib_dir), output_calib_dir, dirs_exist_ok=True)
    print(f"will copy all from {os.path.dirname(src_calib_dir)} to {output_calib_dir}")

    

Select most recent label: /Users/othayothr/data/jarvis_merge/dataset_11_06/calib_params/2024_11_21_17_00_11
will copy from /Users/othayothr/data/jarvis_merge/dataset_11_06/calib_params to /Users/othayothr/data/merged_dataset/calib_params
Select most recent label: /Users/othayothr/data/jarvis_merge/dataset_11_25/calib_params/2024_12_20_11_48_00
will copy from /Users/othayothr/data/jarvis_merge/dataset_11_25/calib_params to /Users/othayothr/data/merged_dataset/calib_params


#### copy train and val images

In [ ]:
# create parent output directory for calibration files

for image_set in ["train", "val"]:

    # create train + val directories in the parent folder
    output_img_dir = os.path.join(output_dataset, image_set)
    if not os.path.exists(output_img_dir):
        os.makedirs(output_img_dir)
    
    # loop over all datasets    
    for dset in input_datasets:
        
        # find the most recent image dataset
        src_image_dir = os.path.join(src_dir, dset, image_set)
        all_imagesets = glob.glob(src_image_dir + "/*")
        all_imagesets.sort()
        select_most_recent_imageset = all_imagesets[-1]
        # print("Select most recent label: {}".format(select_most_recent_calib))
        src_image_dir = select_most_recent_imageset
        

        print(f"will copy all from {os.path.dirname(src_image_dir)} to {output_img_dir}")
        shutil.copytree(os.path.dirname(src_image_dir), output_img_dir, dirs_exist_ok=True)


    

will copy all from /Users/othayothr/data/jarvis_merge/dataset_11_06/train to /Users/othayothr/data/merged_dataset/train
will copy all from /Users/othayothr/data/jarvis_merge/dataset_11_25/train to /Users/othayothr/data/merged_dataset/train
will copy all from /Users/othayothr/data/jarvis_merge/dataset_11_06/val to /Users/othayothr/data/merged_dataset/val
will copy all from /Users/othayothr/data/jarvis_merge/dataset_11_25/val to /Users/othayothr/data/merged_dataset/val


#### copy and merge annotations


In [100]:
def merge_json_annotations(json_data):
   
    root_json = {}
    root_json["keypoint_names"] = json_data[0]["keypoint_names"]
    root_json["skeleton"] = json_data[0]["skeleton"]
    root_json["categories"] = json_data[0]["categories"]

    # Initialize the "calibrations" key in root_json if it doesn't exist
    root_json["calibrations"] = {}
    root_json["images"] = []
    root_json["annotations"] = []
    root_json["framesets"] = {}
      
    img_idx_offset = 0 #need to offset the image ids for each dataset
    
    for dsets in json_data:
        
        # calibrations
        calib_key = list(dsets["calibrations"].keys())[0]
        for calib_key, calib_value in dsets["calibrations"].items():
            root_json["calibrations"][calib_key] = calib_value
            
        #images
        for img in dsets["images"]:
            img["id"] = img["id"] + img_idx_offset
            root_json["images"].append(img)
            
        
        #annotations
        for annots in dsets["annotations"]:
            annots["id"] = annots["id"] + img_idx_offset
            annots["image_id"] = annots["image_id"] + img_idx_offset
            root_json["annotations"].append(annots)
            
        # frameset
        for frameset_key, frameset_value in dsets["framesets"].items():
            frameset_value["frames"] = [x + img_idx_offset for x in frameset_value["frames"]]
            root_json["framesets"][frameset_key] = frameset_value
            

        img_idx_offset += len(dsets["images"])
        

    return root_json


# create train + val directories in the parent folder
output_annot_dir = os.path.join(output_dataset, "annotations")
if not os.path.exists(output_annot_dir):
    os.makedirs(output_annot_dir)
    
# for both train and val
for image_set in ["train", "val"]:
    
    json_data = []
    # loop over all datasets    
    for dset in input_datasets:
    
        src_json_file = os.path.join(src_dir, dset, "annotations", f"instances_{image_set}.json")
        with open(src_json_file, 'r') as file:
            json_data.append(json.load(file))
            
    
    merged_json = merge_json_annotations(json_data)        
        
    output_json_file = os.path.join(output_dataset, "annotations", f"instances_{image_set}.json")
    with open(output_json_file, 'w') as file:
        json.dump(merged_json, file, indent=4)
    

[0, 32, 64, 96, 128, 160, 192, 224, 256, 288, 320, 352, 384, 416, 448, 480]
[10, 42, 74, 106, 138, 170, 202, 234, 266, 298, 330, 362, 394, 426, 458, 490]
